**CELL 1 -  Setup**

In [1]:
!pip install scikit-surprise --quiet
!pip install "numpy<2" --quiet
print('xong')

xong


**Cell 2 - Import**

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from surprise import Dataset, Reader, BaselineOnly, accuracy
from surprise.model_selection import GridSearchCV, KFold

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid")

print("Môi trường đã sẵn sàng.")

Môi trường đã sẵn sàng.


**Cell 3 — Load & làm sạch dữ liệu**

In [3]:
USER_COL = 'CustomerID'
ITEM_COL = 'StockCode'

CANDIDATE_PATHS = [
    '/kaggle/input/datasets/quangthai0406/dataset1/final_online_retail_clean.csv',
    '/kaggle/input/dataset1/final_online_retail_clean.csv',
    '/content/final_online_retail_clean.csv',
    './final_online_retail_clean.csv',
    'final_online_retail_clean.csv',
]
FILE_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if FILE_PATH is None:
    print("[DEBUG] Không thấy file ở path mặc định. Liệt kê /kaggle/input:\n")
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.lower().endswith('.csv'):
                print(os.path.join(root, f))
    raise FileNotFoundError("Copy đúng path .csv in ở trên và gán vào FILE_PATH.")

print(f"[INFO] Đang load dữ liệu từ: {FILE_PATH}")
raw = pd.read_csv(FILE_PATH)
raw = raw.dropna(subset=[USER_COL, ITEM_COL])
raw[USER_COL] = raw[USER_COL].astype(float).astype(int).astype(str)
raw[ITEM_COL] = raw[ITEM_COL].astype(str)
print(f"Số dòng giao dịch gốc: {len(raw):,}")

interactions = (
    raw.groupby([USER_COL, ITEM_COL])
       .agg(TotalQuantity=('Quantity', 'sum'),
            TotalRevenue=('Revenue', 'sum'),
            Frequency=('InvoiceNo', 'nunique'))
       .reset_index()
)
print(f"Số cặp (khách hàng, sản phẩm) duy nhất: {len(interactions):,}")

def kcore_filter(data, user_col, item_col, min_u=5, min_i=5, max_iter=10):
    d = data.copy()
    for _ in range(max_iter):
        u_counts = d.groupby(user_col)[item_col].transform('count')
        i_counts = d.groupby(item_col)[user_col].transform('count')
        before = len(d)
        d = d[(u_counts >= min_u) & (i_counts >= min_i)]
        if len(d) == before:
            break
    return d.reset_index(drop=True)

MIN_INTERACTIONS = 5
df = kcore_filter(interactions, USER_COL, ITEM_COL, min_u=MIN_INTERACTIONS, min_i=MIN_INTERACTIONS)
print(f"[FILTER] Sau lọc: {df[USER_COL].nunique():,} khách / {df[ITEM_COL].nunique():,} SP / {len(df):,} cặp")

df['LogInteractionScore'] = np.log1p(df['TotalRevenue'])
RATING_COL = 'LogInteractionScore'

score_min, score_max = float(df[RATING_COL].min()), float(df[RATING_COL].max())
if score_min == score_max:
    score_max += 1.0
print(f"[RATING SCALE] [{score_min:.3f}, {score_max:.3f}]")

[INFO] Đang load dữ liệu từ: /kaggle/input/datasets/quangthai0406/dataset1/final_online_retail_clean.csv
Số dòng giao dịch gốc: 391,286
Số cặp (khách hàng, sản phẩm) duy nhất: 266,253
[FILTER] Sau lọc: 4,074 khách / 3,146 SP / 264,538 cặp
[RATING SCALE] [0.095, 9.717]


**Cell 4 — Build Surprise Dataset**

In [4]:
reader = Reader(rating_scale=(score_min, score_max))
full_dataset = Dataset.load_from_df(df[[USER_COL, ITEM_COL, RATING_COL]], reader)

all_items = df[ITEM_COL].unique().tolist()
all_users = df[USER_COL].unique().tolist()
items_by_user_full = df.groupby(USER_COL)[ITEM_COL].apply(set).to_dict()

print(f"full_dataset: {len(df):,} tương tác | {len(all_users):,} khách | {len(all_items):,} SP")

full_dataset: 264,538 tương tác | 4,074 khách | 3,146 SP


**Cell 5 — GridSearch Baseline (ALS)**

In [5]:
cv_search = KFold(n_splits=3, random_state=RANDOM_SEED)

print("Đang dò tham số cho Baseline (ALS)...")
baseline_param_grid = {
    'bsl_options': {
        'method': ['als'],
        'n_epochs': [10, 20],
        'reg_u': [10, 15],
        'reg_i': [5, 10],
    }
}
gs_baseline = GridSearchCV(BaselineOnly, baseline_param_grid, measures=['rmse', 'mae'], cv=cv_search, n_jobs=-1)
gs_baseline.fit(full_dataset)
best_bsl_options = gs_baseline.best_params['rmse']['bsl_options']
print(f"[BEST PARAMS] {best_bsl_options} | CV RMSE={gs_baseline.best_score['rmse']:.4f}")

Đang dò tham số cho Baseline (ALS)...
[BEST PARAMS] {'method': 'als', 'n_epochs': 20, 'reg_u': 10, 'reg_i': 5} | CV RMSE=0.6873


**Cell 6 — 5-Fold CV đầy đủ: RMSE/MAE (100% dataset) + Ranking (toàn user, toàn catalog)**

In [6]:
K_EVAL = 10

def ndcg_at_k(topk_items, positive_set, k):
    dcg = sum(1.0/np.log2(i+2) for i, it in enumerate(topk_items[:k]) if it in positive_set)
    idcg = sum(1.0/np.log2(i+2) for i in range(min(len(positive_set), k)))
    return dcg/idcg if idcg > 0 else 0.0

def run_full_kfold_evaluation(algo_class, algo_kwargs, model_name, n_folds=5, k_eval=K_EVAL, verbose_every=1000):
    kf = KFold(n_splits=n_folds, random_state=RANDOM_SEED)
    all_true, all_pred = [], []
    precisions, recalls, ndcgs, hits = [], [], [], []
    t_start = time.time()

    for fold_id, (trainset, testset) in enumerate(kf.split(full_dataset), 1):
        algo = algo_class(**algo_kwargs)
        algo.fit(trainset)

        for p in algo.test(testset):
            all_true.append(p.r_ui)
            all_pred.append(p.est)

        test_by_user = {}
        for u, i, r in testset:
            test_by_user.setdefault(u, set()).add(i)

        print(f"[{model_name}] Fold {fold_id}/{n_folds}: {len(test_by_user):,} khách hàng x {len(all_items):,} SP...")

        for idx, (uid, positives) in enumerate(test_by_user.items(), 1):
            seen_in_train = items_by_user_full.get(uid, set()) - positives
            candidates = [it for it in all_items if it not in seen_in_train]
            scored = [(it, algo.predict(uid, it).est) for it in candidates]
            scored.sort(key=lambda x: x[1], reverse=True)
            topk_items = [it for it, _ in scored[:k_eval]]

            hit_set = set(topk_items) & positives
            precisions.append(len(hit_set) / k_eval)
            recalls.append(len(hit_set) / len(positives))
            hits.append(1.0 if hit_set else 0.0)
            ndcgs.append(ndcg_at_k(topk_items, positives, k_eval))

            if idx % verbose_every == 0:
                print(f"    ...{idx:,}/{len(test_by_user):,} khách (fold {fold_id}) | {(time.time()-t_start)/60:.1f} phút")

    rmse_full = float(np.sqrt(np.mean((np.array(all_true) - np.array(all_pred))**2)))
    mae_full = float(np.mean(np.abs(np.array(all_true) - np.array(all_pred))))

    return {
        "Model": model_name,
        "RMSE_full_dataset": round(rmse_full, 4),
        "MAE_full_dataset": round(mae_full, 4),
        "Users_Evaluated": len(precisions),
        f"Precision@{k_eval}": round(np.mean(precisions), 4),
        f"Recall@{k_eval}": round(np.mean(recalls), 4),
        f"HitRate@{k_eval}": round(np.mean(hits), 4),
        f"NDCG@{k_eval}": round(np.mean(ndcgs), 4),
        "Eval_Time_min": round((time.time()-t_start)/60, 1),
    }

print("="*72 + "\nBASELINE (ALS) — 5-Fold CV đầy đủ (~2-3 phút)\n" + "="*72)
baseline_full_eval = run_full_kfold_evaluation(
    BaselineOnly, {"bsl_options": best_bsl_options, "verbose": False}, "Baseline (ALS)")

df_full_eval = pd.DataFrame([baseline_full_eval])
display(df_full_eval)

BASELINE (ALS) — 5-Fold CV đầy đủ (~2-3 phút)
[Baseline (ALS)] Fold 1/5: 3,961 khách hàng x 3,146 SP...
    ...1,000/3,961 khách (fold 1) | 0.1 phút
    ...2,000/3,961 khách (fold 1) | 0.2 phút
    ...3,000/3,961 khách (fold 1) | 0.3 phút
[Baseline (ALS)] Fold 2/5: 3,954 khách hàng x 3,146 SP...
    ...1,000/3,954 khách (fold 2) | 0.6 phút
    ...2,000/3,954 khách (fold 2) | 0.7 phút
    ...3,000/3,954 khách (fold 2) | 0.8 phút
[Baseline (ALS)] Fold 3/5: 3,949 khách hàng x 3,146 SP...
    ...1,000/3,949 khách (fold 3) | 1.0 phút
    ...2,000/3,949 khách (fold 3) | 1.2 phút
    ...3,000/3,949 khách (fold 3) | 1.3 phút
[Baseline (ALS)] Fold 4/5: 3,930 khách hàng x 3,146 SP...
    ...1,000/3,930 khách (fold 4) | 1.5 phút
    ...2,000/3,930 khách (fold 4) | 1.6 phút
    ...3,000/3,930 khách (fold 4) | 1.7 phút
[Baseline (ALS)] Fold 5/5: 3,974 khách hàng x 3,146 SP...
    ...1,000/3,974 khách (fold 5) | 2.0 phút
    ...2,000/3,974 khách (fold 5) | 2.1 phút
    ...3,000/3,974 khách (fold 5) 

,Model,RMSE_full_dataset,MAE_full_dataset,Users_Evaluated,Precision@10,Recall@10,HitRate@10,NDCG@10,Eval_Time_min
0,Baseline (ALS),0.6827,0.5145,19768,0.0054,0.0062,0.0523,0.005,2.3


**Cell 7 — Breakdown RMSE/MAE theo từng fold**

In [7]:
def get_fold_level_metrics(algo_class, algo_kwargs, model_name, n_folds=5):
    kf = KFold(n_splits=n_folds, random_state=RANDOM_SEED)
    fold_rows = []
    for fold_id, (trainset, testset) in enumerate(kf.split(full_dataset), 1):
        t0 = time.time()
        algo = algo_class(**algo_kwargs)
        algo.fit(trainset)
        preds = algo.test(testset)
        fit_time = time.time() - t0
        y_true = np.array([p.r_ui for p in preds])
        y_pred = np.array([p.est for p in preds])
        rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
        mae = float(np.mean(np.abs(y_true - y_pred)))
        fold_rows.append({"Model": model_name, "Fold": f"Fold {fold_id}",
                           "RMSE": round(rmse, 4), "MAE": round(mae, 4),
                           "Fit Time (s)": round(fit_time, 2)})
        print(f"[{model_name}] Fold {fold_id}/{n_folds} xong ({fit_time:.1f}s)")
    return pd.DataFrame(fold_rows)

baseline_fold_details = get_fold_level_metrics(
    BaselineOnly, {"bsl_options": best_bsl_options, "verbose": False}, "Baseline (ALS)")
display(baseline_fold_details)

[Baseline (ALS)] Fold 1/5 xong (1.6s)
[Baseline (ALS)] Fold 2/5 xong (1.0s)
[Baseline (ALS)] Fold 3/5 xong (1.0s)
[Baseline (ALS)] Fold 4/5 xong (1.5s)
[Baseline (ALS)] Fold 5/5 xong (1.2s)


,Model,Fold,RMSE,MAE,Fit Time (s)
0,Baseline (ALS),Fold 1,0.6843,0.5150,1.61
1,Baseline (ALS),Fold 2,0.6845,0.5155,1.03
2,Baseline (ALS),Fold 3,0.6830,0.5150,1.04
3,Baseline (ALS),Fold 4,0.6819,0.5138,1.54
4,Baseline (ALS),Fold 5,0.6797,0.5134,1.21


**Cell 8 — Train model cuối trên 100% dữ liệu**

In [8]:
full_trainset = full_dataset.build_full_trainset()

baseline_final = BaselineOnly(bsl_options=best_bsl_options, verbose=False)
baseline_final.fit(full_trainset)

print(f"Đã huấn luyện Baseline trên toàn bộ {full_trainset.n_ratings:,} tương tác "
      f"({full_trainset.n_users:,} khách x {full_trainset.n_items:,} SP).")

Đã huấn luyện Baseline trên toàn bộ 264,538 tương tác (4,074 khách x 3,146 SP).


**Cell 9 — API gợi ý**

In [9]:
def recommend_for_user(customer_id: str, k: int = 10, model=baseline_final, exclude_seen: bool = True) -> pd.DataFrame:
    customer_id = str(customer_id)
    try:
        model.trainset.to_inner_uid(customer_id)
    except ValueError:
        return None
    seen = items_by_user_full.get(customer_id, set()) if exclude_seen else set()
    candidates = [it for it in all_items if it not in seen]
    scored = [(it, model.predict(customer_id, it).est) for it in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    return pd.DataFrame(scored[:k], columns=["StockCode", "PredictedScore"])

**Cell 10 — Cold-Start Fallback (Popularity + FP-Growth)**

In [11]:
# ==============================================================================
# CELL 10: XỬ LÝ COLD-START — KHÁCH HÀNG/SẢN PHẨM MỚI CHƯA TỪNG CÓ TRONG TRAIN
# ==============================================================================
import ast

# ------------------------------------------------------------------------------
# A) POPULARITY FALLBACK — dùng khi KHÔNG có bất kỳ thông tin gì (khách hoàn
#    toàn mới, chưa có giỏ hàng, chưa có lịch sử) -> gợi ý top sản phẩm bán chạy.
# ------------------------------------------------------------------------------
popularity_ranking = (
    df.groupby(ITEM_COL)
      .agg(TotalRevenue=('TotalRevenue', 'sum'), NumBuyers=(USER_COL, 'nunique'))
      .sort_values('NumBuyers', ascending=False)
      .reset_index()
)

def recommend_popular(k: int = 10, exclude_items: set = None) -> pd.DataFrame:
    exclude_items = exclude_items or set()
    result = popularity_ranking[~popularity_ranking[ITEM_COL].isin(exclude_items)].head(k)
    return result[[ITEM_COL, 'NumBuyers', 'TotalRevenue']].reset_index(drop=True)

# ------------------------------------------------------------------------------
# B) FP-GROWTH FALLBACK — dùng khi có giỏ hàng hiện tại (VD khách mới nhưng đã
#    thêm vài SP vào giỏ) -> gợi ý SP thường được mua kèm dựa trên luật kết hợp.
# ------------------------------------------------------------------------------
FPGROWTH_CANDIDATE_PATHS = [
    "/kaggle/input/datasets/quangthai0406/dataset1/fpgrowth_rules (2).csv",
    "/kaggle/input/dataset1/fpgrowth_rules (2).csv",
    "./fpgrowth_rules (2).csv",
    "fpgrowth_rules (2).csv",
]
FPGROWTH_RULES_PATH = next((p for p in FPGROWTH_CANDIDATE_PATHS if os.path.exists(p)), None)

df_rules = None
if FPGROWTH_RULES_PATH is not None:
    def parse_frozenset_str(s):
        """Parse "frozenset({'A', 'B'})" -> {'A','B'} an toàn bằng ast.literal_eval."""
        inner = s.strip()[len('frozenset('):-1]
        return ast.literal_eval(inner)

    df_rules = pd.read_csv(FPGROWTH_RULES_PATH)
    df_rules['antecedents_set'] = df_rules['antecedents'].apply(parse_frozenset_str)
    df_rules['consequents_set'] = df_rules['consequents'].apply(parse_frozenset_str)
    print(f"[INFO] Đã load {len(df_rules):,} luật FP-Growth từ: {FPGROWTH_RULES_PATH}")
else:
    print("[WARN] Không tìm thấy file fpgrowth_rules.csv -> recommend_for_cart() sẽ chỉ dùng popularity fallback.")

def recommend_for_cart(cart_items: list, k: int = 10) -> pd.DataFrame:
    """Gợi ý SP mua kèm dựa trên giỏ hàng hiện tại (dùng cho khách mới - cold-start).
    Ưu tiên luật có antecedents là tập con của cart_items, sắp theo lift giảm dần."""
    cart_set = set(str(x) for x in cart_items)

    if df_rules is None:
        return recommend_popular(k=k, exclude_items=cart_set)

    matched = df_rules[df_rules['antecedents_set'].apply(lambda a: a.issubset(cart_set))].copy()
    if matched.empty:
        return recommend_popular(k=k, exclude_items=cart_set)

    matched = matched.sort_values(['lift', 'confidence'], ascending=False)

    records, seen = [], set(cart_set)
    for _, row in matched.iterrows():
        for item in row['consequents_set']:
            if item in seen:
                continue
            records.append({
                ITEM_COL: item,
                "lift": round(row['lift'], 4),
                "confidence": round(row['confidence'], 4),
                "source_rule": " + ".join(sorted(row['antecedents_set'])),
            })
            seen.add(item)
            if len(records) >= k:
                break
        if len(records) >= k:
            break

    result = pd.DataFrame(records)
    if len(result) < k:
        n_missing = k - len(result)
        backup = recommend_popular(k=n_missing, exclude_items=seen)
        backup = backup.rename(columns={'NumBuyers': 'lift', 'TotalRevenue': 'confidence'})
        backup['source_rule'] = "popularity_fallback"
        result = pd.concat([result, backup], ignore_index=True)
    return result

# ------------------------------------------------------------------------------
# C) recommend_for_user PHIÊN BẢN CÓ TỰ ĐỘNG FALLBACK COLD-START
# ------------------------------------------------------------------------------
def recommend_for_user_safe(customer_id: str, k: int = 10, model=baseline_final, cart_items: list = None) -> pd.DataFrame:
    """Gọi hàm này thay cho recommend_for_user() thông thường:
    - Nếu customer_id đã có trong train -> dùng model CF (Baseline) như bình thường.
    - Nếu customer_id là cold-start (chưa từng có trong train):
        + Có cart_items -> dùng FP-Growth fallback (recommend_for_cart).
        + Không có gì cả -> dùng Popularity fallback (recommend_popular).
    """
    result = recommend_for_user(customer_id, k=k, model=model)
    if result is not None:
        return result  # khách đã biết -> dùng model CF bình thường

    print(f"[COLD-START] Khách hàng '{customer_id}' chưa có trong tập train -> dùng fallback.")
    if cart_items:
        return recommend_for_cart(cart_items, k=k)
    return recommend_popular(k=k)

# ------------------------------------------------------------------------------
# Demo nhanh
# ------------------------------------------------------------------------------
print("\n[DEMO] Khách mới hoàn toàn, không có giỏ hàng -> Popularity fallback:")
display(recommend_for_user_safe(customer_id="COLD_USER_999", k=5))

if df_rules is not None:
    sample_cart = [df[ITEM_COL].iloc[0]]
    print(f"\n[DEMO] Khách mới, đang có SP '{sample_cart[0]}' trong giỏ -> FP-Growth fallback:")
    display(recommend_for_cart(sample_cart, k=5))

[INFO] Đã load 100 luật FP-Growth từ: /kaggle/input/datasets/quangthai0406/dataset1/fpgrowth_rules (2).csv

[DEMO] Khách mới hoàn toàn, không có giỏ hàng -> Popularity fallback:
[COLD-START] Khách hàng 'COLD_USER_999' chưa có trong tập train -> dùng fallback.


,StockCode,NumBuyers,TotalRevenue
0,22423,869,140644.75
1,85123A,849,89418.95
2,47566,703,65178.53
3,84879,675,56331.91
4,22720,640,33298.30



[DEMO] Khách mới, đang có SP '16008' trong giỏ -> FP-Growth fallback:


,StockCode,NumBuyers,TotalRevenue
0,22423,869,140644.75
1,85123A,849,89418.95
2,47566,703,65178.53
3,84879,675,56331.91
4,22720,640,33298.30
